In [12]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
import json
import re
import fitz
import pathlib

In [13]:
def organize_wiki(html, url, chunk_size, count=0):
    soup = BeautifulSoup(html, "lxml")

    title = soup.title.text.strip()
    title = re.sub(r'[\\/:*?"<>|]', "_", title)


    data = []

    chunk_id = 0

    current_section = "Introduction"

    section_text = ""

    content = soup.select("h2, h3, p")

    for tag in content:

        # 遇到新 section
        if tag.name in ["h2", "h3"]:

            # flush 上一个 section
            if len(section_text) > 30:

                for i in range(0, len(section_text), chunk_size):

                    chunk = section_text[i:i+chunk_size]

                    data.append({
                        "title": title,
                        "url": url,
                        "section": current_section,
                        "chunk_id": chunk_id,
                        "text": chunk
                    })

                    chunk_id += 1

            # 更新 section
            current_section = tag.get_text(" ", strip=True)

            current_section = re.sub(r"\[[^\]]+\]", "", current_section)

            section_text = ""

        elif tag.name == "p":

            text = tag.get_text(" ", strip=True)

            text = re.sub(r"\[[^\]]+\]", "", text)

            text = " ".join(text.split())

            if len(text) < 30:
                continue

            section_text += "\n" + text

    # 最后一个 section flush
    if len(section_text) > 30:

        for i in range(0, len(section_text), chunk_size):

            chunk = section_text[i:i+chunk_size]

            data.append({
                "title": title,
                "url": url,
                "section": current_section,
                "chunk_id": chunk_id,
                "text": chunk
            })

            chunk_id += 1

    path = f"C:\\Users\\12942\\Desktop\\anlp-spring2026-hw2\\data_process\\data\\{title}.json"
    with open(path, "w+") as f:
        f.write(json.dumps(data, indent=2, ensure_ascii=False))
    print(f"{count} data save in {path}, len : {len(data)}")

In [14]:
def crawler_and_organize_wiki(url, chunk_size, count=0):
    driver = webdriver.Edge()
    driver.get(url)

    time.sleep(5)

    html = driver.page_source

    organize_wiki(html, url, chunk_size, count)

    driver.quit()

In [16]:
from pathlib import Path

folder = Path(r"C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\baseline_data")

# 获取所有 .htm 文件
htm_files = list(folder.glob("*.htm"))

for i, file in enumerate(htm_files):
    with open(file, "r", encoding="utf-8") as f:
        html = f.read()
        url = f"file:///{file.resolve()}"
        organize_wiki(html, url, chunk_size=1000, count=i)

0 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Andrew Carnegie - Wikipedia.json, len : 77
1 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Andrew Mellon - Wikipedia.json, len : 48
2 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Association of American Universities - Wikipedia.json, len : 11
3 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Association of Independent Technological Universities - Wikipedia.json, len : 2
4 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Astrobotic Technology - Wikipedia.json, len : 11
5 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Carnegie Library of Pittsburgh - Wikipedia.json, len : 5
6 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Carnegie Mellon College of Engineering - Wikipedia.json, len : 5
7 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_proces

In [17]:
def crawler_pdf(path, url, chunk_size):
    doc = fitz.open(path)

    title = "2025-operating-budget"
    # url = "https://www.pittsburghpa.gov/files/assets/city/v/4/omb/documents/operating-budgets/2025-operating-budget.pdf"

    chunk_overlap = 128

    toc = doc.get_toc()

    data = []

    global_chunk_id = 0


    def clean_text(text):

        text = re.sub(r"\s+", " ", text)

        text = re.sub(r"\[[^\]]+\]", "", text)

        return text.strip()


    for idx, item in enumerate(toc):

        level, current_section, start_page = item

        # 当前 section 的结束页
        if idx + 1 < len(toc):
            end_page = toc[idx + 1][2] - 1
        else:
            end_page = len(doc)

        print(f"[INFO] {current_section}")
        print(f"       pages: {start_page} -> {end_page}")

        section_text = ""

        # PDF 页码从 0 开始
        for page_num in range(start_page - 1, end_page):

            page = doc[page_num]

            blocks = page.get_text("blocks")

            # 按阅读顺序排序
            blocks = sorted(blocks, key=lambda b: (b[1], b[0]))

            for block in blocks:

                text = block[4]

                text = clean_text(text)

                # 太短跳过
                if len(text) < 20:
                    continue

                section_text += text + "\n"

        if len(section_text) < 50:
            continue

        start = 0

        local_chunk_id = 0

        while start < len(section_text):

            end = start + chunk_size

            chunk = section_text[start:end]

            # 避免截断句子
            if end < len(section_text):

                last_period = chunk.rfind(".")

                last_newline = chunk.rfind("\n")

                split_pos = max(last_period, last_newline)

                if split_pos > chunk_size * 0.5:
                    chunk = chunk[:split_pos + 1]
                    end = start + split_pos + 1

            chunk = chunk.strip()

            if len(chunk) > 30:

                data.append({
                    "title": title,
                    "url": url,
                    "section": current_section,
                    "level": level,
                    "page_start": start_page,
                    "page_end": end_page,
                    "chunk_id": global_chunk_id,
                    "local_chunk_id": local_chunk_id,
                    "text": chunk
                })

                global_chunk_id += 1
                local_chunk_id += 1

            # overlap
            start = end - chunk_overlap


    path = f"{title}.json"

    with open(path, "w", encoding="utf-8") as f:

        json.dump(data, f, indent=2, ensure_ascii=False)

    print("\n=========================")
    print(f"Saved in {path}")
    print(f"Total chunks: {len(data)}")
    print("=========================")

    doc.close()

In [18]:
url = "https://www.pittsburghpa.gov/files/assets/city/v/4/omb/documents/operating-budgets/2025-operating-budget.pdf"
path = r"C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\baseline_data\2025-operating-budget.pdf"
crawler_pdf(path, url, chunk_size=1000)

[INFO] Cover
       pages: 1 -> 1
[INFO] Council - Budget Recognition
       pages: 2 -> 2
[INFO] Table of Contents
       pages: 3 -> 4
[INFO] Budget Legislation
       pages: 5 -> 9
[INFO] The American Rescue Plan
       pages: 10 -> 10
[INFO] ARP Details
       pages: 11 -> 13
[INFO] Exhibit A
       pages: 14 -> 16
[INFO] Budget Guide - Cover Page
       pages: 17 -> 17
[INFO] Overview
       pages: 18 -> 26
[INFO] Budget Cycle
       pages: 27 -> 28
[INFO] Five-Year Financial Forecast - Cover Page
       pages: 29 -> 29
[INFO] Five-Year Financial Forecast
       pages: 30 -> 31
[INFO] Revenue - Cover Page
       pages: 32 -> 32
[INFO] Revenue Summary
       pages: 33 -> 38
[INFO] Revenues Narrative
       pages: 39 -> 43
[INFO] Revenues Detail
       pages: 44 -> 47
[INFO] Expenditures and Departmental Sections
       pages: 48 -> 48
[INFO] Expenditures Summary
       pages: 49 -> 54
[INFO] Expenditures Detail
       pages: 55 -> 57
[INFO] City Council - Cover Page
       pages: 5

In [22]:
with open("C:\\Users\\12942\\Desktop\\anlp-spring2026-hw2\\data\\The Pittsburgh Cultural Trust.html", "r", encoding="utf-8") as f:
    html = f.read()
    url = f"file:///{file.resolve()}"
    organize_wiki(html, url, chunk_size=1000, count=i)

50 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\The Pittsburgh Cultural Trust.json, len : 5


In [19]:
with open(r"C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\extract_web.txt", "r") as f:
    data = f.readlines()

In [20]:
for i, url in enumerate(data):
    crawler_and_organize_wiki(url, chunk_size=1000, count=i)

0 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Pittsburgh - Wikipedia.json, len : 111
1 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\History of Pittsburgh - Wikipedia.json, len : 66
2 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Trash & Recycling - Pittsburgh, PA.json, len : 2
3 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\311 - Pittsburgh, PA.json, len : 3
4 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Reserve Park Shelter - Pittsburgh, PA.json, len : 3
5 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Permits, Licenses, and Inspections - Pittsburgh, PA.json, len : 16
6 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Parks - Pittsburgh, PA.json, len : 20
7 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\City Meeting Hub - Pittsburgh, PA.json, len : 2
8 data save in 

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=60930): Read timed out. (read timeout=120)

In [22]:
for i, url in enumerate(data[26:]):
    crawler_and_organize_wiki(url, chunk_size=1000, count=i+26)

26 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\List of museums in Pittsburgh - Wikipedia.json, len : 1
27 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Pittsburgh Food Festivals - Greek - Oktoberfest - Apples - Maple Syrup _ Visit Pittsburgh.json, len : 18
28 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\🥒 Picklesburgh_ It's A Really Big Dill.json, len : 2
29 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Pittsburgh Taco Festival _ Saturday, September 6th, 2025.json, len : 1
30 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Pittsburgh Restaurant Week – Putting Pittsburgh Restaurants in the spotlight.json, len : 1
31 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\Little Italy Days _ Italian Festival.json, len : 7
32 data save in C:\Users\12942\Desktop\anlp-spring2026-hw2\data_process\data\The Great American Banana Split Ce